# 03 · Observational Causal Inference

Randomized experiments aren't always possible. When treatment assignment isn't controlled, naive comparisons are biased by confounding. Propensity score methods can recover causal estimates if the selection mechanism is observable.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import expit
from scipy.spatial import KDTree
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(42)

plt.rcParams.update({
    'figure.figsize': (9, 4),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

In [ ]:
feats = [f'f{i}' for i in range(12)]
df = pd.read_csv(
    'data/criteo-research-uplift-v2.1.csv',
    usecols=['treatment', 'visit'] + feats
)

# Ground truth from the full randomized dataset
true_ate = df.loc[df.treatment==1,'visit'].mean() - df.loc[df.treatment==0,'visit'].mean()
print(f"Ground truth ATE (full randomized data): {true_ate:.3%}")

## Simulating an observational study

The Criteo data is randomized, so to demonstrate confounding, resample it with selection bias: high-f9 users are overrepresented in the treatment group, low-f9 users in control. Since f9 also correlates strongly with visit (+0.45), this creates a classic omitted variable problem.

In [ ]:
f9_z = (df['f9'] - df['f9'].mean()) / df['f9'].std()

# Selection weights: treated users drawn from high-f9, controls from low-f9
# scale=0.3 gives clear confounding while preserving common support for matching
w_treat = expit( 0.3 * f9_z)
w_ctrl  = expit(-0.3 * f9_z)

n_obs = 20_000
obs_t = df[df.treatment==1].sample(n_obs, weights=w_treat[df.treatment==1].values, random_state=42)
obs_c = df[df.treatment==0].sample(n_obs, weights=w_ctrl [df.treatment==0].values, random_state=42)
obs   = pd.concat([obs_t, obs_c]).reset_index(drop=True)
del df

naive_ate = obs.loc[obs.treatment==1,'visit'].mean() - obs.loc[obs.treatment==0,'visit'].mean()

print(f"Ground truth ATE: {true_ate:.3%}")
print(f"Naive (biased):   {naive_ate:.3%}  ({naive_ate/true_ate:.1f}x the true effect)")
print(f"\nMean f9  |  treatment: {obs.loc[obs.treatment==1,'f9'].mean():.3f}  "
      f"control: {obs.loc[obs.treatment==0,'f9'].mean():.3f}  (should match in RCT, doesn't here)")

## Propensity score estimation

The propensity score $e(x) = P(T=1 \mid X=x)$ summarizes all the confounders into a single number. Conditioning on it is sufficient to remove bias (under the *unconfoundedness* assumption: no unmeasured confounders).

Estimate it via logistic regression on all available features.

In [ ]:
X = obs[feats].values
y = obs['treatment'].values

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X, y)
obs['pscore'] = lr.predict_proba(X)[:, 1].clip(0.02, 0.98)

print(f"Propensity score range: [{obs['pscore'].min():.3f}, {obs['pscore'].max():.3f}]")

fig, ax = plt.subplots()
for t, label, color in [(1,'Treatment','steelblue'), (0,'Control','tomato')]:
    ax.hist(obs.loc[obs.treatment==t,'pscore'], bins=50, alpha=0.6, label=label, color=color, density=True)
ax.set_xlabel('Propensity score')
ax.set_title('Propensity score distributions (overlap = common support)')
ax.legend()
plt.tight_layout()
plt.show()

Good overlap means both methods below are valid. If the distributions barely overlap, units in one group have no comparable counterparts in the other and estimates become unreliable.

## Inverse probability weighting (IPW)

Reweight each unit by the inverse of its probability of receiving the treatment it actually received. This creates a synthetic population where treatment is unrelated to X.

Using the Hajek estimator (normalized weights) for stability.

In [ ]:
t_mask = obs.treatment == 1
c_mask = obs.treatment == 0

w_t = 1 / obs.loc[t_mask, 'pscore']
w_c = 1 / (1 - obs.loc[c_mask, 'pscore'])

ipw_treat = (obs.loc[t_mask,'visit'] * w_t).sum() / w_t.sum()
ipw_ctrl  = (obs.loc[c_mask,'visit'] * w_c).sum() / w_c.sum()
ipw_ate   = ipw_treat - ipw_ctrl

print(f"IPW ATE: {ipw_ate:.3%}")

## Propensity score matching (PSM)

For each treated unit, find the nearest control unit by propensity score. Compute the average difference in outcomes across matched pairs. This estimates the ATT (average treatment effect on the treated).

In [ ]:
treated = obs[t_mask].reset_index(drop=True)
control = obs[c_mask].reset_index(drop=True)

tree = KDTree(control['pscore'].values.reshape(-1, 1))
_, idx = tree.query(treated['pscore'].values.reshape(-1, 1), k=1)

matched_ctrl = control.iloc[idx.flatten()]
psm_att = treated['visit'].values.mean() - matched_ctrl['visit'].values.mean()

# Balance check: f9 before and after matching
f9_imbalance_raw     = obs.loc[t_mask,'f9'].mean() - obs.loc[c_mask,'f9'].mean()
f9_imbalance_matched = treated['f9'].mean() - matched_ctrl['f9'].mean()

print(f"PSM ATT: {psm_att:.3%}")
print(f"f9 imbalance  before matching: {f9_imbalance_raw:.3f}")
print(f"f9 imbalance  after  matching: {f9_imbalance_matched:.3f}")

## Results

In [ ]:
labels   = ['Ground truth', 'Naive', 'IPW', 'PSM (ATT)']
estimates = [true_ate, naive_ate, ipw_ate, psm_att]

fig, ax = plt.subplots(figsize=(8, 3.5))
colors = ['green', 'tomato', 'steelblue', 'steelblue']
for i, (lab, est, col) in enumerate(zip(labels, estimates, colors)):
    ax.barh(i, est * 100, color=col, alpha=0.75)
    ax.text(est * 100 + 0.02, i, f'{est:.3%}', va='center', fontsize=9)

ax.axvline(true_ate * 100, color='green', linestyle='--', linewidth=1, alpha=0.6)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.set_xlabel('Estimated visit lift (pp)')
ax.set_title('Naive vs. adjusted estimates')
plt.tight_layout()
plt.show()

print(f"{'Method':<16} {'Estimate':>10} {'Error vs truth':>15}")
for lab, est in zip(labels, estimates):
    print(f"{lab:<16} {est:>10.3%} {est-true_ate:>+15.3%}")

## Key takeaways

- Naive comparison overestimates the effect because treated units have higher f9, which independently drives visits. Classic omitted variable bias.
- IPW and PSM both recover estimates close to the ground truth by controlling for the propensity score.
- Both methods rest on **unconfoundedness**: all confounders are measured. If something drives both treatment selection and the outcome but isn't in X, neither method fixes it.
- PSM balance check (f9 imbalance before vs. after) is a useful diagnostic. If balance doesn't improve, the model or covariates are wrong.
- **Difference-in-differences** is the third major observational method. It requires panel data (same units observed pre and post) and a parallel trends assumption. Not applicable to this cross-sectional dataset.